# Having an LLM generate RSL, then driving a robot (Kamibot) with that RSL - step-by-step tutorial

In this document, **without using tools (function calls)**, the LLM generates in one shot a program
in **RSL (Robot Scripting Language)** - a language dedicated to robot control. That RSL is then
**compiled (parsed and validated)** with **[`pykamirsl`](https://pypi.org/project/pykamirsl/)** and
**interpreted and executed** to drive the educational robot **Kamibot (KamibotPi)**.

> Comparison: in [`robot-agent-tutorial.md`](robot-agent-tutorial.md), the LLM called the
> `@tool` functions directly for every action (the agent + tool-calling approach).
> This document is different. **The LLM outputs only a single block of RSL source code**,
> and we compile and run it on our side.

## 0. Why RSL? (the difference from tool-calling)

| Aspect | tool-calling agent | **RSL approach (this document)** |
|------|----------------------|------------------------|
| What the LLM does | decides on a tool call per action | **writes one whole RSL program** |
| Who executes | the agent runtime calls tools one by one | **the RSL interpreter runs it after validation** |
| When it is validated | right before each call (effectively never) | **the whole program is checked before execution via `validate()`** |
| Loops / conditions | the LLM re-decides every time | expressed with RSL's `LOOP` / `IF` (no LLM re-calls) |
| Safety | hard to stop if the LLM runs away | loop cap of 100; unsupported commands blocked |
| Reproducibility | can differ on every run | **the same RSL = always the same behavior** |


Overall flow:

```
User natural-language command
      |
      v
 [ LLM ]  --(generates RSL source text per the SYSTEM_PROMPT instructions)--+
      |                                                                     |
      v                                                                     v
 RSL source code  -->  [ parse() ]  -->  [ validate() ]  -->  [ RSLInterpreter.arun() ]
   (text)              parse (AST)       validate (check errors)   interpret & run line by line
                                          | if there are errors                  | tool calls
                                          +-> re-request from the LLM            v
                                              via build_retry_message      [ the real Kamibot robot ]
```

Key idea: **LLM -> RSL -> (compile) -> robot**. The LLM never touches the robot *directly*.

---

## 1. The RSL language at a glance

RSL is a bundle of human-readable, one-line commands. The grammar supported by `pykamirsl`:

```
BEGIN MISSION "title"
  ... commands ...
END MISSION

SET name = value                      # constant assignment only (no arithmetic)
MOVE FORWARD|BACKWARD CM=v            # or SECONDS=v / SPEED=v
TURN LEFT|RIGHT ANGLE=v
STOP
WAIT SECONDS=v
LED RGB r,g,b                         # or LED PRESET n
BEEP
MELODY SCALE=s SECONDS=t
DRAW <shape> SIZE=v                    # triangle, rectangle, pentagon, hexagon, star, circle
SENSE DISTANCE|LINE_*|COLOR|BATTERY AS var
SIGNAL "message {var}"                # a message to show to a person
IF condition: ... END IF              # condition: only <  >  <=  >=  ==  !=
LOOP UNTIL|WHILE condition: ... END LOOP   # at most 100 times (MAX_LOOP_ITERS)
REPEAT count TIMES: ... END REPEAT
```

Example RSL:

```
BEGIN MISSION "Avoid the obstacle"
  SET threshold = 30
  SENSE DISTANCE AS d
  IF d > threshold:
    MOVE BACKWARD CM=10
    BEEP
  END IF
END MISSION
```

> Constraints (safeguards): **no arithmetic operators**, comparison operators only (`< > <= >= == !=`),
> `SET` assigns constants only, **loops are capped at 100 iterations**, and unsupported commands are
> filtered out by `validate()`.


## 2. Installation

In [2]:
!pip install -U langchain "langchain[openai]" pykamilab pykamirsl

## 3. Setting the API key

Do not hardcode the key in your code; keep it in an **environment variable**.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

## 4. Connecting RSL commands to the real robot (the tool adapter)

In [10]:
# robot_tools.py
import asyncio
from pykamilab import KamibotPi

class RobotTool:
    """AsyncTool protocol: only needs a `name` attribute + an async `ainvoke(args)`."""
    def __init__(self, name, fn):
        self.name = name
        self._fn = fn

    async def ainvoke(self, args: dict):
        # pykamilab is synchronous -> hand it to a thread so the event loop isn't blocked
        return await asyncio.to_thread(self._fn, args)


def build_robot_tools(bot: KamibotPi) -> dict:
    """Map standard RSL tool names -> actual Kamibot actions."""
    tools = {
        # --- movement ---
        "move_forward_cm":  lambda a: bot.move_forward_unit(a["distance_cm"], "-l"),
        "move_backward_cm": lambda a: bot.move_backward_unit(a["distance_cm"], "-l"),
        "turn_left_degrees":  lambda a: bot.turn_left_speed(a["degrees"], speed=100),
        "turn_right_degrees": lambda a: bot.turn_right_speed(a["degrees"], speed=100),
        "stop_now":     lambda a: bot.stop(),
        "wait_seconds": lambda a: bot.delay(a["seconds"]),
        # --- sound ---
        "play_beep":        lambda a: bot.beep(),
        "play_melody_note": lambda a: bot.melody(a["scale"], a["seconds"]),
        # --- shapes ---
        "draw_triangle":  lambda a: bot.draw_tri(a["size"]),
        "draw_rectangle": lambda a: bot.draw_rect(a["size"]),
        "draw_pentagon":  lambda a: bot.draw_penta(a["size"]),
        "draw_hexagon":   lambda a: bot.draw_hexa(a["size"]),
        "draw_star":      lambda a: bot.draw_star(a["size"]),
        "draw_circle":    lambda a: bot.draw_circle(a["size"]),
        # --- sensors (return values back into RSL variables) ---
        "read_object_distance_raw": lambda a: bot.get_object_detect(),
        "read_line_sensors_raw":    lambda a: bot.get_line_sensor(),
        "read_battery_level_raw":   lambda a: bot.get_battery(),
    }
    return {name: RobotTool(name, fn) for name, fn in tools.items()}

## 5. Having the LLM generate RSL

The `SYSTEM_PROMPT` instructs the model to "take natural language and output **only RSL**".
We give that as the system message, pass the user's command as-is, and just take the **response text (= RSL)**.


In [11]:
from openai import OpenAI
from pykamrsl import SYSTEM_PROMPT

client = OpenAI()   # uses the OPENAI_API_KEY environment variable

def generate_rsl(user_command: str, extra: str = "") -> str:
    """Natural-language command -> RSL source code (string)."""
    resp = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT + extra},  # on retry, append the error message
            {"role": "user", "content": user_command},
        ],
    )
    return resp.choices[0].message.content.strip()

## 6. Compile + (on failure) automatic retry

When `validate()` returns errors, `build_retry_message(errors)` tells the LLM **what was wrong**
so it can fix and resend the RSL. This is the heart of the "LLM -> RSL -> compile" loop.

In [12]:
from pykamrsl import parse, validate, ParseError, build_retry_message

def compile_rsl(user_command: str, max_retries: int = 2):
    """Natural language -> RSL -> return the (validated) mission."""
    extra = ""
    for attempt in range(max_retries + 1):
        source = generate_rsl(user_command, extra)
        print(f"\n--- RSL generated by the LLM (attempt {attempt + 1}) ---\n{source}\n")
        try:
            mission = parse(source)          # ParseError on a syntax error
            errors = validate(mission)       # list of semantic errors
        except ParseError as e:
            errors = [str(e)]
            mission = None
        if not errors:
            return mission                   # compiled successfully
        # failed -> append the errors to the system prompt and retry
        extra = "\n\n" + build_retry_message(errors)
        print(f"Validation failed -> retrying: {errors}")
    raise RuntimeError("RSL compilation failed repeatedly.")

## 7. Putting it all together - LLM -> RSL -> compile -> robot

In [16]:
# run_rsl_robot.py
import asyncio
from openai import OpenAI
from pykamilab import KamibotPi
from pykamrsl import (
    parse, validate, ParseError,
    RSLInterpreter, SYSTEM_PROMPT, build_retry_message,
)

PORT = "COM69"
client = OpenAI()


def generate_rsl(user_command: str, extra: str = "") -> str:
    resp = client.chat.completions.create(
        model="gpt-4o", max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT + extra},
            {"role": "user", "content": user_command},
        ],
    )
    return resp.choices[0].message.content.strip()


def compile_rsl(user_command: str, max_retries: int = 2):
    extra = ""
    for attempt in range(max_retries + 1):
        source = generate_rsl(user_command, extra)
        print(f"\n=== RSL generated by the LLM (attempt {attempt + 1}) ===\n{source}\n")
        try:
            mission = parse(source)
            errors = validate(mission)
        except ParseError as e:
            mission, errors = None, [str(e)]
        if not errors:
            return mission
        print("Validation failed ->", errors)
        extra = "\n\n" + build_retry_message(errors)
    raise RuntimeError("RSL compilation failed")

In [18]:
command = "Go forward 10 cm, then turn right 90 degrees, beep, and draw a triangle."

# 1) LLM -> RSL,  2) parse + validate (compile)
mission = compile_rsl(command)

# 3) Run the validated RSL on the real robot
bot = KamibotPi(PORT, response_timeout=3.0)
bot.init()


=== RSL generated by the LLM (attempt 1) ===
BEGIN MISSION "Forward Turn Beep Triangle"
  MOVE FORWARD CM=10
  TURN RIGHT ANGLE=90
  BEEP
  DRAW TRIANGLE SIZE=10
END MISSION

KamibotPi Connect PORT=COM69, BAUD=57600


In [19]:

try:
    tools = build_robot_tools(bot)
    log = await RSLInterpreter(tools).arun(mission)   # * asyncio.run -> await
    print("\n=== Execution log ===")
    for line in log:
        print(line)
finally:
    bot.disconnect()   # safely close the port no matter what

[CMD] move_forward_cm(distance_cm=10)
[CMD] turn_right_degrees(degrees=90)
[CMD] play_beep()
[CMD] draw_triangle(size=10)

=== Execution log ===
=== Mission start: Forward Turn Beep Triangle ===
None
None
None
None
=== Mission end ===
Disconnect(KamibotPi) COM69
